# 17. Audit failures and decide what the manuscript can claim

Which conclusions survive the completed experiments, and which attractive explanations remain unsupported? We will inspect actual examples, keep failed gates visible and assemble a claim-by-claim evidence trail for draft02.

## Run this notebook independently

From the repository root, install the shared environment and start Jupyter:

```bash
uv sync --project experiments/aaai
AAAI_ITERATION02_RUN_DIR=/absolute/path/to/experiments/aaai/artifacts/iteration02/PROTOCOL_HASH \
  uv run --project experiments/aaai jupyter lab experiments/aaai
```

Replace `PROTOCOL_HASH` with the exact completed run you intend to inspect. The setup displays its producer command and verifies its recorded identity before showing results. It never chooses the newest run automatically. Alternatively, put `AAAI_ITERATION02_RUN_DIR` in the repository root `.env`. Do not create another environment file. Python-dotenv calls the real functions `load_dotenv` and `find_dotenv`; the aliases below implement the requested `load_env(find_env())` spelling without printing credentials.

Python 3.12 and the notebook, plotting, data and model dependencies are declared in the existing `pyproject.toml`. Select its kernel. This notebook does not depend on cells from another notebook. A full experiment fits models through the recorded producer command; this tutorial loads those real fitted outputs, checks them and teaches how to interpret them. Missing required outputs stop execution instead of triggering synthetic examples.

A small development run has its own protocol identity. To inspect it deliberately, also set `AAAI_ITERATION02_ALLOW_DEVELOPMENT=1`. Its outputs remain labeled incomplete. Do not use that option to bypass a failed evidence verifier.

The experiment is exploratory because the available sources informed earlier work. A dataset folder label is not a diagnosis independently established here. The focal endpoint is normal versus Parkinsons, stroke, cerebral palsy and myopathic labels. The mandatory broad endpoint uses all non-normal folders. The word “normal” refers to the dataset annotation, not a clinical clearance.

In [ ]:
from pathlib import Path
import os, sys, json
from dotenv import load_dotenv as load_env, find_dotenv as find_env

ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents)
            if (p / "ambient").is_dir() and (p / "data/gavd").is_dir())
os.chdir(ROOT)
load_env(find_env())
BASE = ROOT / "experiments/aaai"
if str(BASE) not in sys.path:
    sys.path.insert(0, str(BASE))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display, Markdown
from aaai_gait.iteration02_notebook_support import (
    EvidenceRun, package_versions, show_diagram, show_video, show_status,
    optional_evidence, joint_table, source_role_table, plot_counts,
    prediction_example, example_losses, plot_prediction, plot_adequacy,
    plot_timing, plot_source_scores, within_fold_metrics, plot_metric_table,
    fold_json_records, fold_tables, cohort_table, mask_bank_summary,
    measured_window, plot_paired_differences, timing_video_example, external_replication, plot_condition_confusions,
)

run = EvidenceRun.from_env()
display(run.describe().T.rename(columns={0: "value"}))
display(package_versions())
if run.mode not in {"full", "full_study"}:
    display(Markdown("**DEVELOPMENT RUN. This is incomplete evidence and cannot supply final manuscript results.**"))

## 1. Begin with the verifier, not the abstract

The verification record must confirm the source split, normal-only fitting, frozen mask identities, shared target support, score formulas and metric reconstruction. A passing record supports computational consistency. It cannot certify clinical validity or novelty.

Inspect the exact run identity before copying any number into a slide or manuscript. Development runs, historical draft01 runs and this new protocol are different evidence objects. An unavailable branch stays visible even when the core run completes.

In [ ]:
show_status(run, "verification.json")
show_status(run, "baseline_status.json")
show_diagram("execution_gates")

## 2. Select examples using a declared rule

Use a normal example first, then a labeled abnormal example chosen independently of the score. These show what the source data look like. Error analyses can additionally select high-scoring normal sources or low-scoring abnormal sources, but that outcome-based selection must be stated.

Do not call one selected example representative. Look for tracking errors, aids, turning, camera movement, truncation and unreliable ankle positions. A model failure can be an observation failure, and a highly predictable compensation can produce a biologically abnormal movement with little relational anomaly.

In [ ]:
show_video(run, label="normal")
show_video(run, label=os.getenv("AAAI_PREVIEW_LABEL", "stroke"))

## 3. Inspect outcome-selected failure candidates honestly

The table below deliberately selects the largest normal score and the smallest abnormal score under the U reference. These are diagnostic candidates selected after scoring. They do not define a tuned threshold or prove an error without a threshold rule.

For each candidate, inspect the corresponding source video and quality record. Avoid a clinical story that the visible sequence cannot support. A broad non-normal label can describe an activity or style rather than pathology.

In [ ]:
scores = run.table("source_scores.csv", columns=("source_id", "policy", "bank", "binary_label", "S"))
primary_bank = "U" if "U" in scores.bank.unique() else "E_U"
reference = scores.loc[scores.policy.eq("U") & scores.bank.eq(primary_bank)]
candidates = pd.concat([
    reference.loc[reference.binary_label.eq(0)].nlargest(1, "S").assign(selection="Highest-scoring normal"),
    reference.loc[reference.binary_label.eq(1)].nsmallest(1, "S").assign(selection="Lowest-scoring abnormal"),
], ignore_index=True)
display(candidates)
windows = run.table("window_metadata.csv")
for source in candidates.source_id:
    source_windows = windows.loc[windows.source_id.eq(source)].sort_values(["sequence_id", "start_seconds"])
    if len(source_windows):
        show_video(run, source_windows.iloc[0])
display(run.table("descriptors_quality16.csv"))
display(cohort_table(run, "media_audit.csv"))
show_status(run, "view_sensitivity.json")

## 4. Separate a failure to learn from a failure of the hypothesis

An inadequate local predictor cannot establish what extra context contains. A both-leg correction that fails on normal development cannot support a neurological interpretation of its anomaly score. A timing probe with almost no paired support cannot show timing independence. A source-selection bias can make both successful and failed scores misleading.

The frozen fallback retains those folds with R = 0 and preserves the M ranking. Report how much of the final evaluation actually uses a supported relational component. Do not describe fallback performance as a successful relational model.

In [ ]:
adequacy = run.table("adequacy.csv")
display(adequacy)
fallback = scores.fallback.astype(str).str.lower().isin(["true", "1"]) if "fallback" in scores else None
if fallback is not None:
    display(scores.assign(fallback_flag=fallback).groupby(["policy", "bank", "outer_fold"]).agg(
        sources=("source_id", "nunique"), fallback_fraction=("fallback_flag", "mean")))
paired_adequacy = run.table("timing/paired_support_adequacy.csv")
display(paired_adequacy)
print("Passed sparse-input gates:", int(paired_adequacy.paired_support_adequacy_passed.sum()), "of", len(paired_adequacy))
if not paired_adequacy.paired_support_adequacy_passed.all():
    display(Markdown("**The sparse paired-support adequacy gate failed for at least one comparison. Its timing effect is an input-sensitivity result, not evidence of physiological coordination.**"))

## 5. Use the novelty boundary as a reviewer would

Normal-only masked gait modeling, clinical coordination descriptors, dependence-based anomaly scores and anatomical masking all have close precedents. A JEPA label or a different mask picture is not enough. The possible contribution is a controlled measurement of added information that survives capacity, support, timing and local-motion alternatives.

If simple ridge performs well, the paper can emphasize the measurement and its controls. If a learned encoder does not improve a common coordinate target over a random encoder, report that limitation. If only the focused diagnostic improves, frame the result as task-specific. The study should explain why its observation matters rather than enlarge the claim to match an intended title.

In [ ]:
show_status(run, "historical_common_source_sensitivity.json")
show_status(run, "per_condition_metrics.json")
print("These subgroup and historical checks are descriptive. The draft02 readiness review records the publication claims.")

## 6. Choose the manuscript outcome from the evidence

There are three defensible routes. If added information improves a competent predictor and improves detection beyond M, write a constrained relational-anomaly paper with paired uncertainty. If a reliable relationship exists but detection does not improve, write a measurement study explaining what local and relational views capture differently. If a key estimator or support gate fails, a clearly scoped failure audit may be the honest result, provided the investigation is substantial.

None of these outcomes proves a neural controller, lesion location or causal disease pathway. GAVD lacks the forces, EMG, strength measurements, medication state and clinical examination needed for those claims. A physical world-model program would require additional observations that distinguish competing explanations.

In [ ]:
metrics = run.table("metrics.csv")
display(metrics)
display(pd.json_normalize(run.json("paired_intervals.json")))
display(plot_paired_differences(run)); plt.close("all")
external = external_replication(run)
display(pd.DataFrame([external["completion"]]))
display(pd.DataFrame([{"score": name, "weighted_auroc": values["weighted_auroc"]}
                      for name, values in external["metrics"].items()]))
display(pd.json_normalize(external["paired"]))
display(pd.json_normalize(external["age_sex"]))
print("The independent run is separately fitted on simultaneous laboratory angles.")

## 7. Understand what the independent replication adds

The linked public laboratory study evaluates 128 controls and 50 stroke participants, using six simultaneously sampled joint angles and its own normal references. It does not transfer GAVD weights or pool web videos with laboratory participants. The opposite-leg correction improves held-out normal prediction in each fold, but the fixed M/R fusion still separates the groups less well than M alone.

An age and sex sensitivity uses a rule fixed before its outcomes: same-sex pairs within five years, matched without replacement inside each outer fold. Forty-five pairs remain. Whole-pair bootstrap resampling preserves the match. This comparison is a demographic sensitivity analysis, not causal adjustment, and it still favors local movement. It supports a narrow lesson: useful normal predictive dependence does not automatically become useful anomaly evidence.

The article's participant metadata supply age and sex. These values are not guessed from videos or names. The [independent replication report](docs/iteration02/external_replication_results.md) documents the cohort exclusions, synchronized signals and separate fitting. Neither dataset identifies a lesion, muscle force or clinical mechanism.

## 8. Finish the publication evidence package systematically

The manuscript should include an attrition table, physiology-to-observation diagram, access diagram, normal-adequacy plot, paired timing plot, matched baseline table and representative failures. Every measured figure needs its run identity and source artifacts. Illustrations remain illustrations, and unavailable analyses remain unavailable.

Check that the README, notebooks, slides, protocol, results and draft02 use the same source counts, endpoints, mask names, score definitions and intervals. Keep the historical draft01 intact. An independently executed notebook record and a rendered manuscript are necessary readiness checks, but submission format and scientific strength are separate judgments. The final readiness report should state both plainly.

In [ ]:
figure_records = sorted(run.figure_dir.glob("*.json"))
display(pd.DataFrame([json.loads(path.read_text()) for path in figure_records]))
print("Measured figure provenance records:", len(figure_records))

## Preserve the evidence trail

The run directory is the unit of reproducibility. Keep its input hashes, source roles, fitted models, score references, target predictions, source scores, verification record and failed-comparison ledger together. A successful notebook execution establishes that this tutorial can reproduce its displayed analysis. It does not establish a neurological cause or a clinically valid classifier.

The [iteration02 protocol](docs/iteration02/protocol.md), [results](docs/iteration02/results.md), [limitations](docs/iteration02/limitations.md), and [draft02 readiness review](docs/draft02/README.md) must agree with the same run. The earlier draft01 and notebooks 00–11 remain historical evidence. New figures are written as SVG and PDF masters with PNG previews and an input-hash sidecar under this run's `figures/notebooks` directory.

In [ ]:
record_path = run.record('17_failure_analysis_and_manuscript_decision')
print("Recorded the exact real artifacts read by this notebook:", record_path)
print("This notebook did not replace the experiment producer or alter its source roles.")